The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from imports import *
from preprocessing_util import get_column_groups, get_transformers
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_recall_curve, f1_score, average_precision_score, roc_auc_score
%load_ext autoreload
%autoreload 2

In [3]:
query = "select * from train_data"
train_data = sql_connect(query)
X = train_data.drop(columns=['TARGET'])
y = train_data['TARGET']
del train_data
X.head()

Connection to SQL Server established successfully.
Connection closed.


,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,DEBT_TO_INCOME,AVERAGE_EXTERNAL_RATING,N_DOCUMENTS_PROVIDED,ADDITIONAL_DOC_PROVIDED,TOT_PREV_APP,APPROVED_RATIO,REFUSED_RATIO,CANCELLED_RATIO,UNUSED_RATIO,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_RATING_CLIENT,YEARS_BIRTH,YEARS_EMPLOYED,YEARS_REGISTRATION,YEARS_ID_PUBLISH,YEARS_LAST_PHONE_CHANGE,ORG_GROUP,OCCUPATION_TYPE_GROUPED,EDUCATION_LEVEL
0,F,0,0,0,0.909091,0.4045,1,True,0,0.0,0.0,0.0,0.0,Unaccompanied,Working,Single / not married,House / apartment,2,54.91,1.88,25.36,9.68,1.22,Self-Employed,WhiteCollar/Admin,Higher Academic
1,M,0,1,0,5.334971,0.1513,1,True,0,0.0,0.0,0.0,0.0,Unaccompanied,Working,Married,House / apartment,2,26.01,1.14,0.67,5.97,4.41,Self-Employed,Labour/LowSkill,Medium Education
2,F,0,1,0,3.938567,0.3395,1,True,0,0.0,0.0,0.0,0.0,Unaccompanied,Pensioner,Separated,House / apartment,2,56.44,-0.00,23.97,1.66,3.21,Unknown/Other,NonActive,Medium Education
3,F,0,1,0,2.231920,0.7154,1,True,0,0.0,0.0,0.0,0.0,Unaccompanied,Working,Widow,House / apartment,2,65.42,6.66,0.94,11.83,-0.00,Unknown/Other,Technical/Skilled,Medium Education
4,F,1,1,0,5.084746,0.2553,1,True,0,0.0,0.0,0.0,0.0,Unaccompanied,Working,Married,House / apartment,2,32.68,3.83,9.24,5.12,1.30,Public Service,WhiteCollar/Admin,Medium Education


In [4]:
column_groups = get_column_groups(X)

column_groups

{'log_col': ['DEBT_TO_INCOME',
  'YEARS_EMPLOYED',
  'YEARS_LAST_PHONE_CHANGE',
  'YEARS_REGISTRATION'],
 'scale_only_cols': ['TOT_PREV_APP',
  'N_DOCUMENTS_PROVIDED',
  'REGION_RATING_CLIENT',
  'CNT_CHILDREN',
  'REFUSED_RATIO',
  'CANCELLED_RATIO',
  'AVERAGE_EXTERNAL_RATING',
  'UNUSED_RATIO',
  'YEARS_BIRTH',
  'APPROVED_RATIO',
  'YEARS_ID_PUBLISH'],
 'binary_cols': ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'ADDITIONAL_DOC_PROVIDED'],
 'categorical_cols': ['CODE_GENDER',
  'NAME_TYPE_SUITE',
  'NAME_INCOME_TYPE',
  'NAME_FAMILY_STATUS',
  'NAME_HOUSING_TYPE',
  'ORG_GROUP',
  'OCCUPATION_TYPE_GROUPED',
  'EDUCATION_LEVEL']}

In [5]:
transformers = get_transformers(
    column_groups['log_col'],
    column_groups['scale_only_cols'],
    column_groups['categorical_cols'],
    column_groups['binary_cols']
)

In [15]:
preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')
preprocessor

,transformers,"[('log_scale_pipeline', ...), ('binary_passthrough', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,func,<ufunc 'log1p'>
,inverse_func,None
,validate,True


In [7]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_strategy

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [7]:
preprocessor.set_output(transform='pandas')
preprocessor.fit_transform(X).head()

c:\Users\adars\Documents\Ireland_Project\Loan_approval_ml\loan_approval_env\Lib\site-packages\sklearn\preprocessing\_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
c:\Users\adars\Documents\Ireland_Project\Loan_approval_ml\loan_approval_env\Lib\site-packages\sklearn\preprocessing\_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62
0,-1.700371,-0.366466,-0.408920,1.023438,0.0,0.0,1.0,-0.354729,0.198282,-0.099158,-0.582108,-0.15127,-0.139927,-0.699441,-0.048817,0.922065,-0.303132,0.355851,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.755985,-0.673024,0.892023,-2.159182,0.0,1.0,1.0,-0.354729,0.198282,-0.099158,-0.582108,-0.15127,-0.139927,-2.391916,-0.048817,-1.491944,-0.303132,-0.538125,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.246040,-1.458356,0.525745,0.960948,0.0,1.0,1.0,-0.354729,0.198282,-0.099158,-0.582108,-0.15127,-0.139927,-1.133923,-0.048817,1.049865,-0.303132,-1.576680,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,-0.622262,0.643290,-1.573692,-1.986309,0.0,1.0,1.0,-0.354729,0.198282,-0.099158,-0.582108,-0.15127,-0.139927,1.378720,-0.048817,1.799962,-0.303132,0.873924,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.673455,0.167258,-0.357215,-0.067279,1.0,1.0,1.0,-0.354729,0.198282,-0.099158,-0.582108,-0.15127,-0.139927,-1.696745,-0.048817,-0.934801,-0.303132,-0.742945,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
cat_encoder = preprocessor.named_transformers_['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(column_groups['categorical_cols'])
cat_feature_names
col_names = column_groups['log_col'] + column_groups['binary_cols'] + column_groups['scale_only_cols'] + cat_feature_names.tolist()

array(['CODE_GENDER_F', 'CODE_GENDER_M', 'NAME_TYPE_SUITE_Unspecified',
       'NAME_TYPE_SUITE_Family', 'NAME_TYPE_SUITE_Group of people',
       'NAME_TYPE_SUITE_Children', 'NAME_TYPE_SUITE_Unaccompanied',
       'NAME_TYPE_SUITE_Other_A', 'NAME_TYPE_SUITE_Other_B',
       'NAME_INCOME_TYPE_Student', 'NAME_INCOME_TYPE_Unemployed',
       'NAME_INCOME_TYPE_Pensioner',
       'NAME_INCOME_TYPE_Commercial associate',
       'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Working',
       'NAME_INCOME_TYPE_Maternity leave', 'NAME_FAMILY_STATUS_Married',
       'NAME_FAMILY_STATUS_Separated',
       'NAME_FAMILY_STATUS_Civil marriage',
       'NAME_FAMILY_STATUS_Single / not married',
       'NAME_FAMILY_STATUS_Widow', 'NAME_HOUSING_TYPE_Office apartment',
       'NAME_HOUSING_TYPE_With parents',
       'NAME_HOUSING_TYPE_House / apartment',
       'NAME_HOUSING_TYPE_Municipal apartment',
       'NAME_HOUSING_TYPE_Co-op apartment', 'ORG_GROUP_Transport',
       'ORG_GROUP_Healthcare',